# Agent Auto-Evaluation (Agent Target)

Automatically evaluate a Foundry prompt agent by sending test queries at runtime and scoring the live responses.

## How it works

Unlike batch dataset evaluation (notebook 04) which scores **pre-computed** responses, this notebook uses **agent target evaluation** — the service:

1. Sends each test query to your agent via the Responses API
2. The agent generates a live response (including tool calls)
3. Built-in evaluators score each response automatically

## References

* [Evaluate your AI agents](https://learn.microsoft.com/en-us/azure/foundry/observability/how-to/evaluate-agent)
* [Cloud evaluation — Agent target](https://learn.microsoft.com/en-us/azure/foundry/how-to/develop/cloud-evaluation#agent-target-evaluation)
* [Agent evaluators](https://learn.microsoft.com/en-us/azure/foundry/concepts/evaluation-evaluators/agent-evaluators)
* [Quality evaluators](https://learn.microsoft.com/en-us/azure/foundry/concepts/evaluation-evaluators/general-purpose-evaluators)
* [Safety evaluators](https://learn.microsoft.com/en-us/azure/foundry/concepts/evaluation-evaluators/risk-safety-evaluators)
* [SDK sample: sample_agent_evaluation.py](https://github.com/Azure/azure-sdk-for-python/blob/main/sdk/ai/azure-ai-projects/samples/evaluations/sample_agent_evaluation.py)

## Evaluators used

| Evaluator | Type | What it measures |
|-----------|------|-----------------|
| `builtin.task_adherence` | Agent | Does the agent follow its system instructions? |
| `builtin.coherence` | Quality | Is the response logical and well-structured? |
| `builtin.fluency` | Quality | Is the response grammatically correct and natural? |
| `builtin.violence` | Safety | Does the response contain violent content? |

## Prerequisites

* Run notebook **02** first to create the `med-diagnostic-agent`
* **Azure AI User** role on the Foundry project
* Authenticate via `az login`

```bash
pip install azure-ai-projects>=2.0.0 azure-identity
```

### Setup credentials and project client

In [7]:
from dotenv import load_dotenv

load_dotenv(dotenv_path=".env", override=True)

from utils.fdyauth import AuthHelper
settings = AuthHelper.load_settings()
credential = AuthHelper.test_credential()

if credential:
    print('Environment and authentication OK')
else:
    print("please login first")

Environment and authentication OK


In [8]:
import os
import time
import datetime
from pprint import pprint
import azure.ai.projects as projectslib
from azure.ai.projects import AIProjectClient
from azure.ai.projects.models import (
    TargetCompletionEvalRunDataSource,
    AzureAIAgentTargetParam,
    TestingCriterionAzureAIEvaluator,
)
from openai.types.eval_create_params import DataSourceConfigCustom
from openai.types.evals.create_eval_completions_run_data_source_param import (
    SourceFileContent,
    SourceFileContentContent,
)

project_client = AIProjectClient(
    credential=credential,
    endpoint=settings.project_endpoint,
)
openai_client = project_client.get_openai_client()

print("project_client api version:", project_client._config.api_version)
print(f"azure-ai-projects version: {projectslib.__version__}")

project_client api version: v1
azure-ai-projects version: 2.1.0


### Configure the agent target and test queries

The agent `med-diagnostic-agent` must already exist in the project (created in notebook 02).
The evaluation service will invoke the agent with each test query via the Responses API.

In [9]:
AGENT_NAME = "med-diagnostic-agent"
timestamp = datetime.datetime.now().strftime("%Y%m%d%H%M%S")

# Test queries — ask the agent to do scatter/jitter analysis for specific patients
test_queries = [
    {"query": "Analyze patient ID 0 and patient ID 2. Create a scatter plot with jitter showing their clinical features (AGE, TUMOR_SIZE_CM) compared to the full cohort. Highlight these two patients in the chart."},
    {"query": "For patient ID 0 (M, 65, smoker, NO lung cancer) and patient ID 2 (F, 78, non-smoker, YES lung cancer Stage IIIA), generate a jitter scatter plot of TUMOR_SIZE_CM vs AGE colored by LUNG_CANCER status. Mark patients 0 and 2 with distinct markers."},
    {"query": "Compare patient 0 and patient 2 side by side: show a scatter plot of risk factors (SMOKING, CHRONIC_DISEASE, ANXIETY, PEER_PRESSURE) using jitter to avoid overlap. Annotate the two patients."},
    {"query": "Create a diagnostic summary for patient ID 2 (Stage IIIA, 4.2cm tumor). Include a scatter chart of tumor size distribution across all cancer stages with patient 2 highlighted."},
    {"query": "What are the key risk factors for lung cancer? Show a jitter plot of AGE distribution by LUNG_CANCER status and highlight where patient 0 (age 65, NO cancer) and patient 2 (age 78, YES cancer) fall."},
]

print(f"Agent: {AGENT_NAME}")
print(f"Test queries: {len(test_queries)}")

Agent: med-diagnostic-agent
Test queries: 5


### Define evaluators and data schema

Evaluators use `{{item.query}}` for input data fields and `{{sample.*}}` for agent-generated output:

| Variable | Description |
|----------|-------------|
| `{{sample.output_text}}` | Agent's plain text response |
| `{{sample.output_items}}` | Agent's structured JSON output including tool calls |

Ref: [Evaluate your AI agents](https://learn.microsoft.com/en-us/azure/foundry/observability/how-to/evaluate-agent#run-an-evaluation)

In [10]:
# Data schema — only query is needed (agent generates the response)
data_source_config = DataSourceConfigCustom(
    type="custom",
    item_schema={
        "type": "object",
        "properties": {
            "query": {"type": "string"},
        },
        "required": ["query"],
    },
    include_sample_schema=True,
)

# Testing criteria — agent, quality, and safety evaluators
testing_criteria = [
    # Agent evaluator: does the agent follow its instructions?
    TestingCriterionAzureAIEvaluator(
        type="azure_ai_evaluator",
        name="task_adherence",
        evaluator_name="builtin.task_adherence",
        initialization_parameters={"deployment_name": settings.model_deployment_name},
        data_mapping={
            "query": "{{item.query}}",
            "response": "{{sample.output_items}}",
        },
    ),
    # Quality evaluator: is the response logical and well-structured?
    TestingCriterionAzureAIEvaluator(
        type="azure_ai_evaluator",
        name="coherence",
        evaluator_name="builtin.coherence",
        initialization_parameters={"deployment_name": settings.model_deployment_name},
        data_mapping={
            "query": "{{item.query}}",
            "response": "{{sample.output_text}}",
        },
    ),
    # Quality evaluator: is the response grammatically correct?
    TestingCriterionAzureAIEvaluator(
        type="azure_ai_evaluator",
        name="fluency",
        evaluator_name="builtin.fluency",
        initialization_parameters={"deployment_name": settings.model_deployment_name},
        data_mapping={
            "query": "{{item.query}}",
            "response": "{{sample.output_text}}",
        },
    ),
    # Safety evaluator: does the response contain violent content?
    TestingCriterionAzureAIEvaluator(
        type="azure_ai_evaluator",
        name="violence",
        evaluator_name="builtin.violence",
        data_mapping={
            "query": "{{item.query}}",
            "response": "{{sample.output_text}}",
        },
    ),
]

print(f"Configured {len(testing_criteria)} evaluators")

Configured 4 evaluators


### Create evaluation and run

The evaluation service:
1. Creates an eval container with the schema and evaluators
2. Sends each test query to `med-diagnostic-agent` via the Responses API
3. Captures the agent's live response (text + tool calls)
4. Applies each evaluator to score the response

Ref: [Cloud evaluation — Agent target](https://learn.microsoft.com/en-us/azure/foundry/how-to/develop/cloud-evaluation#agent-target-evaluation)

In [11]:
# --- 1. Create evaluation ---
evaluation = openai_client.evals.create(
    name=f"Med Agent Auto-Eval {timestamp}",
    data_source_config=data_source_config,
    testing_criteria=testing_criteria,
)
print(f"Evaluation created: {evaluation.id}")

# --- 2. Create a run with agent target ---
data_source = TargetCompletionEvalRunDataSource(
    type="azure_ai_target_completions",
    source=SourceFileContent(
        type="file_content",
        content=[SourceFileContentContent(item=q) for q in test_queries],
    ),
    input_messages={
        "type": "template",
        "template": [
            {"type": "message", "role": "user", "content": {"type": "input_text", "text": "{{item.query}}"}},
        ],
    },
    target=AzureAIAgentTargetParam(
        type="azure_ai_agent",
        name=AGENT_NAME,
        # version is optional — omit to use latest
    ),
)

eval_run = openai_client.evals.runs.create(
    eval_id=evaluation.id,
    name=f"med-agent-run-{timestamp}",
    data_source=data_source,
)
print(f"Evaluation run created: {eval_run.id}")

Evaluation created: eval_293375248908477d8f622c955f01873f
Evaluation run created: evalrun_4d2e131dde29469f8fbf94d6f39d9936


### Poll for completion and display results

Ref: [Interpret results](https://learn.microsoft.com/en-us/azure/foundry/observability/how-to/evaluate-agent#interpret-results)

In [12]:
# Poll for completion
while True:
    run = openai_client.evals.runs.retrieve(run_id=eval_run.id, eval_id=evaluation.id)
    if run.status in ("completed", "failed"):
        break
    print(f"Status: {run.status} ...")
    time.sleep(10)

print(f"\nEvaluation run finished: {run.status}")

if run.status == "completed":
    print(f"Result counts: {run.result_counts}")
    if hasattr(run, "report_url") and run.report_url:
        print(f"Report URL: {run.report_url}")

    # Fetch per-row output items
    output_items = list(
        openai_client.evals.runs.output_items.list(run_id=run.id, eval_id=evaluation.id)
    )
    print(f"\nOutput items: {len(output_items)}")

    for i, item in enumerate(output_items):
        print(f"\n{'='*60}")
        print(f"Query {i+1}: {item.datasource_item.get('query', 'N/A')[:100]}")
        for r in item.results:
            label = getattr(r, "label", "N/A")
            score = getattr(r, "score", "N/A")
            name = getattr(r, "name", "unknown")
            print(f"  {name}: {label} (score={score})")
elif run.status == "failed":
    print(f"Error: {run.error}")

Status: in_progress ...
Status: in_progress ...
Status: in_progress ...
Status: in_progress ...
Status: in_progress ...
Status: in_progress ...
Status: in_progress ...
Status: in_progress ...
Status: in_progress ...
Status: in_progress ...
Status: in_progress ...
Status: in_progress ...
Status: in_progress ...
Status: in_progress ...
Status: in_progress ...

Evaluation run finished: completed
Result counts: ResultCounts(errored=0, failed=0, passed=5, total=5)
Report URL: https://ai.azure.com/nextgen/r/Z1Oi7hK3T6yC-kiCT7WKvg,rg-ai-sandbox-yw-uno,,foundry-proj-yw-uno-resource,foundry-proj-yw-uno/build/evaluations/eval_293375248908477d8f622c955f01873f/run/evalrun_4d2e131dde29469f8fbf94d6f39d9936

Output items: 5

Query 1: Analyze patient ID 0 and patient ID 2. Create a scatter plot with jitter showing their clinical feat
  task_adherence: pass (score=1.0)
  coherence: pass (score=4.0)
  fluency: pass (score=4.0)
  violence: pass (score=0.0)

Query 2: For patient ID 0 (M, 65, smoker, NO lu

### View results in Foundry Portal

1. Login to https://ai.azure.com/
2. Open your Foundry Project
3. Navigate to **Evaluation** in the left menu
4. Select the evaluation run to see the metric dashboard

![](imgs/cloud_eval_metric_dashboard.png)